# 02 – PINN Training

This notebook trains the Physics-Informed Neural Network (PINN) surrogate on the synthetic dataset generated in `01_data_generation.ipynb`.

The model predicts the three quality metrics from process parameters and spatial coordinates. Training combines:

- **Data loss** (MSE between predictions and synthetic ground truth)
- **Physics residuals** (heat, stress, porosity, geometry)
- **GradNorm-style adaptive loss balancing**

For speed, the architecture and number of epochs are reduced compared with the full default config.

In [ ]:
import os
import sys

# Make the repository's src/ package importable from this notebook
notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, '..'))
sys.path.insert(0, project_root)

# Scripts in src/ expect to be run from the repository root, so switch CWD
os.chdir(project_root)
print(f"Working directory: {os.getcwd()}")
print(f"Project root: {project_root}")

## 2.1 Load and override the configuration for a quick demo

We keep the material properties and optimizer bounds from `data/params.yaml` but shrink the model and training budget.

In [ ]:
import yaml

config_path = os.path.join('data', 'params.yaml')
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Quick-demo overrides
config['model']['hidden_width'] = 64
config['model']['hidden_depth'] = 2
config['training']['n_epochs'] = 5
config['training']['batch_size'] = 16
config['training']['checkpoint_freq'] = 5
config['training']['plot_freq'] = 5
config['training']['print_freq'] = 5

print('Model config:', config['model'])
print('Training config:', config['training'])

## 2.2 Instantiate and run the trainer

`PINNTrainer` accepts an in-memory `config` dict, which lets us apply the overrides above without editing the YAML file.

In [ ]:
from src.pinn.train import PINNTrainer

trainer = PINNTrainer(config_path=config_path, config=config, num_threads=2)

# Train the model
trainer.train()

print(f"\nRun directory: {trainer.run_dir}")
print(f"Best model checkpoint: {trainer.checkpoint_dir / 'best_model.pt'}")

## 2.3 Evaluate on the held-out test set

The `evaluate()` method reports per-output MSE/RMSE/R², physics residual components, and the percentage of predictions that fall outside the physical output bounds.

In [ ]:
metrics = trainer.evaluate()

print('Test-set metrics:')
print(f"  Total MSE: {metrics['test_mse_total']:.6f}")
print(f"  RMSE per output: {metrics['test_rmse_per_output']}")
print(f"  R² per output: {metrics['test_r2_per_output']}")
print(f"  Physics residual: {metrics['test_physics_residual']:.6f}")
print(f"  Bound violations (%): {metrics['test_bound_violations_pct']:.2f}%")

## 2.4 Display training plots

The trainer writes PNG plots under `data/models/<timestamp>/plots/`. We can display them directly from the notebook.

In [ ]:
from IPython.display import Image, display

plot_files = ['loss_curves.png', 'loss_components.png', 'adaptive_weights.png']
for plot_file in plot_files:
    plot_path = trainer.plot_dir / plot_file
    if plot_path.exists():
        print(f"\n{plot_file}")
        display(Image(filename=str(plot_path)))

## Next step

Proceed to `03_optimisation.ipynb` to run NSGA-III (and optionally Bayesian) optimization with the trained checkpoint.